# Set Up

In [ ]:
import pandas as pd
import json
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import xgboost as xgb
import time
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate
import random
import os
import gc
from pathlib import Path
from optuna.integration import XGBoostPruningCallback

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)

# Configuration

In [ ]:
base_dir = Path('/scratch/bng/cartbind/code/MIND_models')
data_dir = Path('/scratch/bng/cartbind/data/UKB_new_data/combined_data_no_outliers')
splits_dir = base_dir / 'scaling_law_splits'
region_dir = base_dir / 'region_names'

hyperparams_dir = base_dir / 'models_xgboost_dnanexus/xgboost_hyperparameters_scaling_law'
results_dir = base_dir / 'models_xgboost_dnanexus/xgboost_scaling_law_results'
predictions_dir = base_dir / 'models_xgboost_dnanexus/xgboost_predictions_scaling_law'
plots_dir = base_dir / 'models_xgboost_dnanexus/xgboost_scaling_law_plots'
hyperparams_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
predictions_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

rename = pd.read_csv(region_dir / 'col_renames_dnanexus.csv')
rename_dict = dict(zip(rename['datafield_code'], rename['datafield_name']))

targets = {
    'GF': ('GF', 'p20016_i2'),
    'PAL': ('PAL', 'p20197_i2'),
    'DSST': ('DSST', 'p23324_i2'),
    'TMT': ('TMT', 'p6350_i2'),
}

data_configs = {
    # 'demo': (None, ['p31', 'p21003_i2', 'p54_i2']),
    # 'MIND_avg': (region_dir / 'MIND_avg_regions.txt', ['p31', 'p21003_i2', 'p54_i2']),
    # 'CT': (region_dir / 'CT_regions_dnanexus.txt', ['p31', 'p21003_i2', 'p54_i2']),
    # 'FC25': (region_dir / 'FC25_regions.txt', ['p31', 'p21003_i2', 'p54_i2', 'p25741_i2']),
    'FC100': (region_dir / 'FC100_regions.txt', ['p31', 'p21003_i2', 'p54_i2', 'p25741_i2']),
    # 'MIND': (region_dir / 'MIND_regions.txt', ['p31', 'p21003_i2', 'p54_i2']),
}

# sample_sizes = [250, 500, 1000, 2000, 4000, 8000, 16000, 32000, 'all']
sample_sizes = ['all']

N_TRIALS = 200
NUM_ROUND = 2000
EARLY_STOPPING_ROUNDS = 50
MIN_RESOURCE = 500

# XGBoost Analysis Function

In [ ]:
def objective(trial, data):
    dtrain, dvalid, y_val = data
    
    # Define the search space
    params = {
        'eta':              trial.suggest_float('eta', 1e-3, 1e-1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 1, 8),
        'min_child_weight': trial.suggest_float('min_child_weight', 3, 70, log=True),
        'subsample':        trial.suggest_float('subsample', 0.1, 0.7),
        'gamma':            trial.suggest_float('gamma', 1e-4, 30, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.2, 0.9),
        'lambda':           trial.suggest_float('lambda', 1e-1, 1000.0, log=True),
        'alpha':            trial.suggest_float('alpha', 1e-1, 1000.0, log=True),
        'eval_metric':      'rmse',
        'objective':        'reg:squarederror',
        'tree_method':      'hist',
        'device':           'cuda',
        'booster':          'gbtree',
        'seed':             seed,
        'verbosity':        0
    }
    
    num_round = NUM_ROUND

    pruning_callback = XGBoostPruningCallback(trial, "eval-rmse")

    model = xgb.train(params, dtrain, num_round,
                      evals=[(dvalid, 'eval')],
                      early_stopping_rounds=EARLY_STOPPING_ROUNDS, 
                      callbacks=[pruning_callback],
                      verbose_eval=False)
    
    preds = model.predict(dvalid)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    
    # Save best_iteration to the trial
    best_iteration = int(model.best_iteration) + 1 if model.best_iteration is not None else num_round
    trial.set_user_attr("best_iteration", best_iteration)
    
    return rmse

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        return super().default(obj)

In [ ]:
def xgboost_analysis(X, y, hyperparams_dir, predictions_dir, plots_dir, 
                     data_name, target_name, sample_size, n_splits=10):
    """
    Nested CV with XGBoost + Optuna.
    """
    outer_cv = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    outer_mae, outer_rmse, outer_r2, outer_r2_corr = [], [], [], []

    target_predictions_dir = predictions_dir / target_name
    target_predictions_dir.mkdir(parents=True, exist_ok=True)
    preds_filename = f'XGBoost_preds_{data_name}_{target_name}_{sample_size}.csv'
    preds_path = target_predictions_dir / preds_filename

    # Fixed parameters to add back to the optimized set
    fixed_params = {
        'eval_metric':      'rmse',
        'objective':        'reg:squarederror',
        'tree_method':      'hist',
        'device':           'cuda',
        'booster':          'gbtree',
        'seed':             seed,
    }

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_test = X.iloc[test_idx].copy()
        y_test = y.iloc[test_idx]

        # Inner split for hyperparam optimization
        X_train, X_val, y_train, y_val = train_test_split(
            X.iloc[train_idx], y.iloc[train_idx], test_size=0.15, random_state=seed
        )

        dtrain_opt = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
        dvalid_opt = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

        # Optuna Optimization
        # Define a lambda that calls our objective with the data
        objective_func = lambda trial: objective(trial, (dtrain_opt, dvalid_opt, y_val))
        
        # Turn off optuna logging to keep output clean or set to ERROR
        optuna.logging.set_verbosity(optuna.logging.WARNING)

        pruner = optuna.pruners.HyperbandPruner(
            reduction_factor=3, # Default is 3, controls how aggressively trials are pruned
            min_resource=MIN_RESOURCE,
            max_resource=NUM_ROUND
        )
                
        study = optuna.create_study(direction='minimize', 
                                    sampler=optuna.samplers.TPESampler(seed=seed+fold),
                                    pruner=pruner)
        study.optimize(objective_func, n_trials=N_TRIALS, show_progress_bar=True)

        pruned_trials = study.get_trials(deepcopy=False, states=[optuna.trial.TrialState.PRUNED])
        complete_trials = study.get_trials(deepcopy=False, states=[optuna.trial.TrialState.COMPLETE])
        num_pruned = len(pruned_trials)
        num_complete = len(complete_trials)
        
        print(f"  Optuna Trials - Complete: {num_complete} | Pruned: {num_pruned}")

        # Create directory for this config and fold
        fold_plot_dir = plots_dir / target_name / data_name / f'n_{sample_size}' / f'fold_{fold}'
        fold_plot_dir.mkdir(parents=True, exist_ok=True)

        # 1. Optimization History
        fig_history = plot_optimization_history(study)
        fig_history.write_html(str(fold_plot_dir / 'optimization_history.html'))
        fig_history.write_image(str(fold_plot_dir / 'optimization_history.png'))

        # 2. Parameter Importances
        try:
            fig_importances = plot_param_importances(study)
            fig_importances.write_html(str(fold_plot_dir / 'param_importances.html'))
            fig_importances.write_image(str(fold_plot_dir / 'param_importances.png'))
        except Exception as e:
            print(f"  Could not plot parameter importances for fold {fold}: {e}")

        # 3. Parallel Coordinate
        fig_parallel = plot_parallel_coordinate(study)
        fig_parallel.write_html(str(fold_plot_dir / 'parallel_coordinate.html'))
        fig_parallel.write_image(str(fold_plot_dir / 'parallel_coordinate.png'))

        del dtrain_opt, dvalid_opt

        # Get best results
        best_trial = study.best_trial
        best_hyperparams = best_trial.params
        optimal_num_round = best_trial.user_attrs['best_iteration']

        # Build training params
        final_params = best_hyperparams.copy()
        final_params.update(fixed_params)

        # Train final model on ONLY the training set
        dtrain_best = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
        dtest_final = xgb.DMatrix(X_test, enable_categorical=True)

        # Train best model on optimal_num_round
        final_model = xgb.train(final_params, dtrain_best, num_boost_round=optimal_num_round, verbose_eval=False)
        y_pred = final_model.predict(dtest_final)

        fold_df = pd.DataFrame({
            'fold': fold,
            'eid': y_test.index,
            'actual': y_test.values,
            'predicted': y_pred
        })
        fold_df.to_csv(preds_path, mode='a', header=(fold == 1), index=False)

        mae  = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2   = r2_score(y_test, y_pred)
        if np.std(y_pred) == 0:
            corr = 0.0
        else:
            corr = np.corrcoef(y_test, y_pred.squeeze())[0, 1]
        r2_corr = corr ** 2

        outer_mae.append(mae)
        outer_rmse.append(rmse)
        outer_r2.append(r2)
        outer_r2_corr.append(r2_corr)

        print(f'  Fold {fold:02d} • MAE={mae:.3f} • RMSE={rmse:.3f} • R²={r2:.3f} • R²(corr)={r2_corr:.3f}')

        # Save fold hyperparameters
        fold_dir = hyperparams_dir / target_name / data_name / f'n_{sample_size}' / f'fold_{fold}'
        fold_dir.mkdir(parents=True, exist_ok=True)

        # Extract feature importances and save to CSV
        importance_dict = final_model.get_score(importance_type='gain')
        if importance_dict:
            importance_df = pd.DataFrame(
                list(importance_dict.items()),
                columns=['Feature', 'Gain']
            ).sort_values(by='Gain', ascending=False)
            
            importance_df.to_csv(fold_dir / 'feature_importances_gain.csv', index=False)

        with (fold_dir / 'best_hyperparameters.json').open('w') as fh:
            json.dump(
                {
                    'best_hyperparams': best_hyperparams,
                    'num_boost_round': optimal_num_round,
                    'best_rmse_val': best_trial.value,
                    'trials_complete': num_complete,
                    'trials_pruned': num_pruned
                },
                fh,
                indent=4,
                cls=NpEncoder
            )

        del fold_df
        del final_model, dtrain_best, dtest_final
        del X_train, X_val, y_train, y_val, X_test, y_test
        gc.collect()

    print(f'\n  Mean MAE    : {np.mean(outer_mae):.3f} ± {np.std(outer_mae):.3f}')
    print(f'  Mean RMSE   : {np.mean(outer_rmse):.3f} ± {np.std(outer_rmse):.3f}')
    print(f'  Mean R²     : {np.mean(outer_r2):.3f} ± {np.std(outer_r2):.3f}')
    print(f'  Mean R²(corr): {np.mean(outer_r2_corr):.3f} ± {np.std(outer_r2_corr):.3f}')

    return {
        'mean_mae':      np.mean(outer_mae),     'std_mae':     np.std(outer_mae),
        'mean_rmse':     np.mean(outer_rmse),    'std_rmse':    np.std(outer_rmse),
        'mean_r2':       np.mean(outer_r2),      'std_r2':      np.std(outer_r2),
        'mean_r2_corr':  np.mean(outer_r2_corr), 'std_r2_corr': np.std(outer_r2_corr),
    }

# Scaling Law Training Loop

In [ ]:
for target_name, (test_key, score_col) in targets.items():
    print(f'\n{"="*60}\nTARGET: {target_name}\n{"="*60}')

    data_file = data_dir / f'combined_data_{test_key}_no_outliers.csv'
    target_splits_dir = splits_dir / target_name

    df_full = pd.read_csv(data_file, index_col=0)

    for data_name, (regions_file, demographic_cols) in data_configs.items():
        print(f'\n--- {target_name} vs. {data_name} ---')

        if regions_file is not None:
            with open(regions_file, 'r') as f:
                brain_regions = [line.strip() for line in f]
        else:
            brain_regions = []

        all_input_cols = demographic_cols + brain_regions

        for sample_size in sample_sizes:
            eid_file = (target_splits_dir / f'{target_name}_all_eids.txt'
                        if sample_size == 'all'
                        else target_splits_dir / f'{target_name}_eids_{sample_size}.txt')

            if not eid_file.exists():
                print(f'  Skipping n={sample_size}: EID file not found.')
                continue

            sample_eids = np.loadtxt(eid_file, dtype=int)
            if 'eid' in df_full.columns:
                df = df_full[df_full['eid'].isin(sample_eids)]
            else:
                df = df_full[df_full.index.isin(sample_eids)]

            actual_n = len(df)
            print(f'\n[{target_name} | {data_name} | n={sample_size} ({actual_n} rows)]')

            X = df[all_input_cols].rename(columns=rename_dict)
            y = df[score_col]

            # Determine variable types after renaming
            categorical_cols = [c for c in ['sex', 'assessment_centre'] if c in X.columns]

            for col in categorical_cols:
                X[col] = X[col].astype('category')

            start_time = time.time()

            try:
                metrics = xgboost_analysis(
                    X, y, hyperparams_dir, predictions_dir, plots_dir, 
                    data_name, target_name, sample_size
                )
            except Exception as e:
                print(f'  Error: {e}')
                continue

            elapsed_sec = time.time() - start_time
            elapsed_min = elapsed_sec / 60
            print(f'  Time for [{data_name}, n={sample_size}]: {elapsed_min:.2f}m')

            results_file = results_dir / f'scaling_law_results_{target_name}.csv'
            row_df = pd.DataFrame([{
                'target_name': target_name,
                'data_name':   data_name,
                'sample_size': sample_size,
                'actual_n':    actual_n,
                **metrics,
                'elapsed_time_min': elapsed_min,
            }])
            row_df.to_csv(results_file, mode='a', header=not results_file.exists(), index=False)

            del df, X, y, metrics
            gc.collect()

print(f'\nDone. Results saved → {results_dir}')